# Spatial Artifacts Detector — Colab GPU Training

Train the two-branch spatial artifacts detector (pixel + frequency) on the DFDC `train_sample_videos` set using a Colab GPU. Checkpoints saved to Google Drive every epoch.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Checkpoint directory on Drive (persists across Colab session disconnects)
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/deepfake-detector/checkpoints"
!mkdir -p "$DRIVE_CHECKPOINT_DIR"

In [ ]:
# Install project dependencies
# Colab is Linux + CUDA, so mediapipe should install and work (unlike local Mac)
!pip install -q torch timm opencv-python pydantic pydantic-settings scikit-learn

# Also install the local package if the repo is cloned in Colab
# !pip install -e .  # uncomment if repo is available

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
from google.colab import userdata

# Kaggle credentials from Colab Secrets (key icon in left sidebar)
# Generated at https://www.kaggle.com/settings/api -> "Create New API Token"
# kagglehub.login() is intentionally NOT called (flaky validation ping bug)
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
print("Kaggle credentials set from Colab Secrets")

In [ ]:
!pip install -q kagglehub
import kagglehub
print("kagglehub", kagglehub.__version__)

In [ ]:
COMPETITION = "deepfake-detection-challenge"
DATA_DIR = "/content/data"
SAMPLE_PREFIX = "train_sample_videos/"

!mkdir -p "$DATA_DIR"

# Download metadata first to verify credentials end-to-end
meta_path = kagglehub.competition_download(
    COMPETITION,
    path=f"{SAMPLE_PREFIX}metadata.json",
    output_dir=DATA_DIR,
)
print("metadata downloaded to:", meta_path)

In [ ]:
import base64
import json
import urllib.parse
import urllib.request


def list_competition_files(competition, prefix):
    """List the competition's data-tab files under `prefix` (paginated)."""
    auth = base64.b64encode(
        f"{os.environ['KAGGLE_USERNAME']}:{os.environ['KAGGLE_KEY']}".encode()
    ).decode()
    names, token = [], None
    while True:
        url = f"https://www.kaggle.com/api/v1/competitions/data/list/{competition}"
        if token:
            url += "?pageToken=" + urllib.parse.quote(token)
        req = urllib.request.Request(url, headers={"Authorization": f"Basic {auth}"})
        data = json.loads(urllib.request.urlopen(req, timeout=60).read())
        names.extend(f["name"] for f in data["files"] if f["name"].startswith(prefix))
        token = data.get("nextPageTokenNullable")
        if not token:
            break
    return sorted(names)


sample_files = list_competition_files(COMPETITION, SAMPLE_PREFIX)
print(f"{len(sample_files)} files in {SAMPLE_PREFIX}")
print(sample_files[:3], "...")

In [ ]:
from tqdm.auto import tqdm

# Download the sample videos one file at a time
for name in tqdm(sample_files):
    if name.endswith("metadata.json"):
        continue  # already downloaded
    kagglehub.competition_download(COMPETITION, path=name, output_dir=DATA_DIR)

print("train_sample_videos download complete")

In [ ]:
# Verify download
!echo "--- structure ---"
!find /content/data/train_sample_videos -type f | head -5
!echo "..."
!echo "--- file count ---"
!find /content/data/train_sample_videos -type f | wc -l
!echo "--- total size ---"
!du -sh /content/data/train_sample_videos

In [ ]:
import json
from pathlib import Path

metadata_path = Path(DATA_DIR) / SAMPLE_PREFIX / "metadata.json"
metadata = json.loads(metadata_path.read_text())
print(f"{len(metadata)} clips in metadata.json\n")

labels = {}
for info in metadata.values():
    labels[info["label"]] = labels.get(info["label"], 0) + 1
print("label distribution:", labels, "\n")

print("First 5 clips (filename -> label / split / original):")
for name, info in list(metadata.items())[:5]:
    print(f"  {name} -> {info}")

## Training Setup

In [ ]:
import random
from collections import defaultdict
import torch
import torch.nn as nn

# Reproducibility
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Load metadata (already downloaded in previous cell)
metadata_path = Path(DATA_DIR) / SAMPLE_PREFIX / "metadata.json"
metadata = json.loads(metadata_path.read_text())

# Identity-aware split: group fakes with their original source clip
real_videos = [k for k, v in metadata.items() if v["label"] == "REAL"]
fake_videos = [k for k, v in metadata.items() if v["label"] == "FAKE"]

real_to_fakes = defaultdict(list)
for fake in fake_videos:
    original = metadata[fake].get("original")
    if original and original in real_videos:
        real_to_fakes[original].append(fake)

# Use ALL available data (not just a small subset)
all_real = real_videos
all_fake = fake_videos

# Split identities 80/20
random.shuffle(all_real)
split_idx = int(0.8 * len(all_real))
train_real = all_real[:split_idx]
val_real = all_real[split_idx:]

train_fake = []
val_fake = []
for real in train_real:
    train_fake.extend(real_to_fakes.get(real, []))
for real in val_real:
    val_fake.extend(real_to_fakes.get(real, []))

# Remaining fakes without a known original -> assign to train
assigned_fakes = set(train_fake + val_fake)
unassigned_fakes = [f for f in fake_videos if f not in assigned_fakes]
train_fake.extend(unassigned_fakes)

train_files = train_real + train_fake
val_files = val_real + val_fake

train_labels = [1 if metadata[v]["label"] == "FAKE" else 0 for v in train_files]
val_labels = [1 if metadata[v]["label"] == "FAKE" else 0 for v in val_files]

print(f"Train: {len(train_files)} clips (REAL: {train_labels.count(0)}, FAKE: {train_labels.count(1)})")
print(f"Val:   {len(val_files)} clips (REAL: {val_labels.count(0)}, FAKE: {val_labels.count(1)})")

# Class-weighted loss: pos_weight = n_neg / n_pos (inverse class frequency)
n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
pos_weight = n_neg / max(n_pos, 1)
print(f"Class pos_weight (FAKE:REAL ratio): {pos_weight:.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device="cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
# Model setup — ported from scripts/train_spatial_artifacts.py (SpatialArtifactsDetector internals)
import timm
import torch.nn as nn

IMAGE_SIZE = 224
FREQUENCY_BINS = 32
HIGH_FREQUENCY_RADIUS_RATIO = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
FACE_CROP_MARGIN = 0.2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

class PixelBranch(nn.Module):
    """Encoder -> pool -> dropout -> linear head."""
    def __init__(self, dropout_rate: float = 0.2) -> None:
        super().__init__()
        self.encoder = timm.create_model("efficientnet_b0", pretrained=True)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(self.encoder.num_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.encoder.forward_features(x)
        x = self.avg_pool(x).flatten(1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


class FrequencyBranch(nn.Module):
    """Small MLP over FFT radial energy distribution + high-frequency ratio."""
    def __init__(
        self,
        in_features: int = FREQUENCY_BINS + 1,
        hidden_dim: int = 64,
        dropout_rate: float = 0.2,
    ) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)

pixel_model = PixelBranch().to(DEVICE)
frequency_model = FrequencyBranch().to(DEVICE)

opt_pixel = torch.optim.AdamW(pixel_model.parameters(), lr=1e-4, weight_decay=1e-5)
opt_freq = torch.optim.AdamW(frequency_model.parameters(), lr=1e-4, weight_decay=1e-5)

print(f"Pixel model params: {sum(p.numel() for p in pixel_model.parameters()):,}")
print(f"Frequency model params: {sum(p.numel() for p in frequency_model.parameters()):,}")


In [ ]:
# Training loop — 10 epochs, checkpoint to Drive every epoch
import cv2
import numpy as np
from torch.utils.data import DataLoader, Dataset

FRAMES_PER_VIDEO = 5
BATCH_SIZE = 8
NUM_EPOCHS = 10

def isotropically_resize_image(img, size, interpolation_down=cv2.INTER_AREA, interpolation_up=cv2.INTER_CUBIC):
    h, w = img.shape[:2]
    if max(w, h) == size:
        return img
    if w > h:
        scale = size / w
        h = h * scale
        w = size
    else:
        scale = size / h
        w = w * scale
        h = size
    interpolation = interpolation_up if scale > 1 else interpolation_down
    return cv2.resize(img, (int(w), int(h)), interpolation=interpolation)

def put_to_center(img, input_size):
    img = img[:input_size, :input_size]
    image = np.zeros((input_size, input_size, 3), dtype=np.uint8)
    start_w = (input_size - img.shape[1]) // 2
    start_h = (input_size - img.shape[0]) // 2
    image[start_h:start_h + img.shape[0], start_w:start_w + img.shape[1], :] = img
    return image

def detect_face_haar(rgb):
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) == 0:
        return None
    x, y, w, h = faces[0]
    return (int(x), int(y), int(w), int(h))

def crop_face(rgb, box):
    x, y, w, h = box
    margin_x, margin_y = int(FACE_CROP_MARGIN * w), int(FACE_CROP_MARGIN * h)
    x0, y0 = max(0, x - margin_x), max(0, y - margin_y)
    x1, y1 = min(rgb.shape[1], x + w + margin_x), min(rgb.shape[0], y + h + margin_y)
    return rgb[y0:y1, x0:x1]

def preprocess_face(crop):
    resized = isotropically_resize_image(crop, IMAGE_SIZE)
    centered = put_to_center(resized, IMAGE_SIZE)
    normalized = (centered.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return np.transpose(normalized, (2, 0, 1))

def extract_frames_and_faces(video_path, num_frames=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return []
    frame_idxs = np.linspace(0, frame_count - 1, num_frames, endpoint=True, dtype=int)
    face_crops = []
    for frame_idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        box = detect_face_haar(rgb)
        if box is not None:
            crop = crop_face(rgb, box)
            face_crops.append(preprocess_face(crop))
    cap.release()
    return face_crops

def frequency_features(gray_crop):
    gray = gray_crop.astype(np.float32)
    gray = gray - gray.mean()
    spectrum = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
    height, width = spectrum.shape
    center_y, center_x = height // 2, width // 2
    yy, xx = np.mgrid[:height, :width]
    radius = np.sqrt((yy - center_y) ** 2 + (xx - center_x) ** 2)
    max_radius = float(np.hypot(center_y, center_x))
    if max_radius <= 0.0:
        return np.zeros(FREQUENCY_BINS + 1, dtype=np.float32)
    bin_edges = np.linspace(0.0, max_radius, FREQUENCY_BINS + 1)
    radial_energy, _ = np.histogram(radius, bins=bin_edges, weights=spectrum)
    total_energy = float(radial_energy.sum())
    distribution = radial_energy / total_energy if total_energy > 0.0 else radial_energy
    high_frequency_mask = radius > HIGH_FREQUENCY_RADIUS_RATIO * max_radius
    high_frequency_energy = float(spectrum[high_frequency_mask].sum())
    high_frequency_ratio = high_frequency_energy / total_energy if total_energy > 0.0 else 0.0
    return np.concatenate([distribution, [high_frequency_ratio]]).astype(np.float32)

class FaceCropDataset(Dataset):
    def __init__(self, video_files, labels, data_dir):
        self.video_files = video_files
        self.labels = labels
        self.data_dir = Path(data_dir)
        self.face_caches = {}
        self._precompute()

    def _precompute(self):
        print("Precomputing face crops...")
        for i, video_file in enumerate(self.video_files):
            video_path = self.data_dir / video_file
            crops = extract_frames_and_faces(video_path)
            if crops:
                self.face_caches[i] = crops
            else:
                self.face_caches[i] = [np.zeros((3, IMAGE_SIZE, IMAGE_SIZE), dtype=np.float32)]
        print(f"Precomputed {len(self.face_caches)} videos")

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        crops = self.face_caches[idx]
        crop = random.choice(crops)
        label = self.labels[idx]
        return torch.from_numpy(crop), torch.tensor(label, dtype=torch.float32)

train_dataset = FaceCropDataset(train_files, train_labels, DATA_DIR + "/" + SAMPLE_PREFIX)
val_dataset = FaceCropDataset(val_files, val_labels, DATA_DIR + "/" + SAMPLE_PREFIX)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

def train_one_epoch(model_pixel, model_freq, loader, opt_pixel, opt_freq, criterion, device):
    model_pixel.train()
    model_freq.train()
    total_loss = 0.0
    num_batches = 0
    for faces, labels in loader:
        faces = faces.to(device)
        labels = labels.to(device).unsqueeze(1)

        opt_pixel.zero_grad()
        opt_freq.zero_grad()

        logits_pixel = model_pixel(faces)
        loss_pixel = criterion(logits_pixel, labels)

        freq_features = torch.stack([
            torch.from_numpy(frequency_features(
                cv2.cvtColor(
                    (face.permute(1, 2, 0).cpu().numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)) * 255,
                    cv2.COLOR_RGB2GRAY
                ).astype(np.uint8)
            )).to(device)
            for face in faces
        ]).float()

        logits_freq = model_freq(freq_features)
        loss_freq = criterion(logits_freq, labels)

        loss = loss_pixel + loss_freq
        loss.backward()
        opt_pixel.step()
        opt_freq.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)

@torch.no_grad()
def validate(model_pixel, model_freq, loader, criterion, device):
    model_pixel.eval()
    model_freq.eval()
    total_loss = 0.0
    num_batches = 0
    for faces, labels in loader:
        faces = faces.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits_pixel = model_pixel(faces)
        loss_pixel = criterion(logits_pixel, labels)

        freq_features = torch.stack([
            torch.from_numpy(frequency_features(
                cv2.cvtColor(
                    (face.permute(1, 2, 0).cpu().numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)) * 255,
                    cv2.COLOR_RGB2GRAY
                ).astype(np.uint8)
            )).to(device)
            for face in faces
        ]).float()

        logits_freq = model_freq(freq_features)
        loss_freq = criterion(logits_freq, labels)

        loss = loss_pixel + loss_freq
        total_loss += loss.item()
        num_batches += 1

    return total_loss / max(num_batches, 1)

print(f"\nStarting training for {NUM_EPOCHS} epochs...")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(pixel_model, frequency_model, train_loader, opt_pixel, opt_freq, criterion, DEVICE)
    val_loss = validate(pixel_model, frequency_model, val_loader, criterion, DEVICE)
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    # Save checkpoint to Drive after every epoch
    ckpt_path = f"{DRIVE_CHECKPOINT_DIR}/spatial_artifacts_epoch{epoch}.pt"
    torch.save({
        "epoch": epoch,
        "pixel_model": pixel_model.state_dict(),
        "frequency_model": frequency_model.state_dict(),
        "pos_weight": pos_weight,
    }, ckpt_path)
    print(f"  Checkpoint saved: {ckpt_path}")

print("\nTraining complete!")


## Evaluation

In [ ]:
# Evaluation — ported from scripts/evaluate_spatial_artifacts.py
import asyncio
from pathlib import Path

# Load the final checkpoint
FINAL_CKPT = f"{DRIVE_CHECKPOINT_DIR}/spatial_artifacts_epoch{NUM_EPOCHS}.pt"
print(f"Loading checkpoint: {FINAL_CKPT}")

checkpoint = torch.load(FINAL_CKPT, map_location=DEVICE, weights_only=True)
pixel_model.load_state_dict(checkpoint["pixel_model"])
frequency_model.load_state_dict(checkpoint["frequency_model"])
pixel_model.eval()
frequency_model.eval()

# Reproduce the exact training selection to know which identities to exclude
from collections import defaultdict
import random

real_videos = [k for k, v in metadata.items() if v["label"] == "REAL"]
fake_videos = [k for k, v in metadata.items() if v["label"] == "FAKE"]

real_to_fakes = defaultdict(list)
for fake in fake_videos:
    original = metadata[fake].get("original")
    if original and original in real_videos:
        real_to_fakes[original].append(fake)

random.seed(42)
selected_real = random.sample(real_videos, min(20, len(real_videos)))
selected_fake = []
for real in selected_real:
    fakes_for_real = real_to_fakes.get(real, [])
    selected_fake.extend(fakes_for_real)

if len(selected_fake) > 20:
    selected_fake = random.sample(selected_fake, 20)
elif len(selected_fake) < 20:
    remaining_fakes = [f for f in fake_videos if f not in selected_fake]
    selected_fake.extend(random.sample(remaining_fakes, min(20 - len(selected_fake), len(remaining_fakes))))

train_identities = set(selected_real)
for fake in selected_fake:
    orig = metadata[fake].get("original")
    if orig:
        train_identities.add(orig)

# Select evaluation clips identity-disjoint from training
EVAL_REAL = 10
EVAL_FAKE = 10

available_real = [v for v in real_videos if v not in train_identities]
available_fake = [v for v in fake_videos if metadata[v].get("original") not in train_identities]

random.seed(43)
eval_real = random.sample(available_real, min(EVAL_REAL, len(available_real)))
eval_fake = random.sample(available_fake, min(EVAL_FAKE, len(available_fake)))

eval_clips = [(v, "REAL") for v in eval_real] + [(v, "FAKE") for v in eval_fake]

print(f"Training identities excluded: {len(train_identities)}")
print(f"Evaluation clips: {len(eval_clips)} ({sum(1 for _, l in eval_clips if l=='REAL')} REAL, {sum(1 for _, l in eval_clips if l=='FAKE')} FAKE)")
print()

# Face detection & preprocessing functions (from training cell)
def isotropically_resize_image(img, size, interpolation_down=cv2.INTER_AREA, interpolation_up=cv2.INTER_CUBIC):
    h, w = img.shape[:2]
    if max(w, h) == size:
        return img
    if w > h:
        scale = size / w
        h = h * scale
        w = size
    else:
        scale = size / h
        w = w * scale
        h = size
    interpolation = interpolation_up if scale > 1 else interpolation_down
    return cv2.resize(img, (int(w), int(h)), interpolation=interpolation)

def put_to_center(img, input_size):
    img = img[:input_size, :input_size]
    image = np.zeros((input_size, input_size, 3), dtype=np.uint8)
    start_w = (input_size - img.shape[1]) // 2
    start_h = (input_size - img.shape[0]) // 2
    image[start_h:start_h + img.shape[0], start_w:start_w + img.shape[1], :] = img
    return image

def detect_face_haar(rgb):
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) == 0:
        return None
    x, y, w, h = faces[0]
    return (int(x), int(y), int(w), int(h))

def crop_face(rgb, box):
    x, y, w, h = box
    margin_x, margin_y = int(FACE_CROP_MARGIN * w), int(FACE_CROP_MARGIN * h)
    x0, y0 = max(0, x - margin_x), max(0, y - margin_y)
    x1, y1 = min(rgb.shape[1], x + w + margin_x), min(rgb.shape[0], y + h + margin_y)
    return rgb[y0:y1, x0:x1]

def preprocess_face(crop):
    resized = isotropically_resize_image(crop, IMAGE_SIZE)
    centered = put_to_center(resized, IMAGE_SIZE)
    normalized = (centered.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return np.transpose(normalized, (2, 0, 1))

def frequency_features(gray_crop):
    gray = gray_crop.astype(np.float32)
    gray = gray - gray.mean()
    spectrum = np.abs(np.fft.fftshift(np.fft.fft2(gray)))
    height, width = spectrum.shape
    center_y, center_x = height // 2, width // 2
    yy, xx = np.mgrid[:height, :width]
    radius = np.sqrt((yy - center_y) ** 2 + (xx - center_x) ** 2)
    max_radius = float(np.hypot(center_y, center_x))
    if max_radius <= 0.0:
        return np.zeros(FREQUENCY_BINS + 1, dtype=np.float32)
    bin_edges = np.linspace(0.0, max_radius, FREQUENCY_BINS + 1)
    radial_energy, _ = np.histogram(radius, bins=bin_edges, weights=spectrum)
    total_energy = float(radial_energy.sum())
    distribution = radial_energy / total_energy if total_energy > 0.0 else radial_energy
    high_frequency_mask = radius > HIGH_FREQUENCY_RADIUS_RATIO * max_radius
    high_frequency_energy = float(spectrum[high_frequency_mask].sum())
    high_frequency_ratio = high_frequency_energy / total_energy if total_energy > 0.0 else 0.0
    return np.concatenate([distribution, [high_frequency_ratio]]).astype(np.float32)

# Fusion weights from config
FUSION_WEIGHT_PIXEL = 0.6
FUSION_WEIGHT_FREQUENCY = 0.4

def confident_strategy(pred, t=0.8):
    pred = np.array(pred)
    sz = len(pred)
    fakes = np.count_nonzero(pred > t)
    if fakes > sz // 2.5 and fakes > 11:
        return np.mean(pred[pred > t])
    elif np.count_nonzero(pred < 0.2) > 0.9 * sz:
        return np.mean(pred[pred < 0.2])
    else:
        return np.mean(pred)

@torch.no_grad()
def analyze_video(video_path):
    cap = cv2.VideoCapture(str(video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None
    
    # Sample 5 frames evenly
    frame_idxs = np.linspace(0, frame_count - 1, 5, endpoint=True, dtype=int)
    face_crops = []
    
    for frame_idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        box = detect_face_haar(rgb)
        if box is not None:
            crop = crop_face(rgb, box)
            face_crops.append(preprocess_face(crop))
    
    cap.release()
    
    if not face_crops:
        return 0.5
    
    # Pixel branch
    batch = torch.stack([torch.from_numpy(c) for c in face_crops]).to(DEVICE)
    logits_pixel = pixel_model(batch)
    probs_pixel = torch.sigmoid(logits_pixel).squeeze(-1).cpu().numpy()
    
    # Frequency branch
    freq_feats = torch.stack([
        torch.from_numpy(frequency_features(cv2.cvtColor(
            (c.permute(1, 2, 0).numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)) * 255,
            cv2.COLOR_RGB2GRAY
        ).astype(np.uint8))).to(DEVICE)
        for c in batch
    ]).float()
    logits_freq = frequency_model(freq_feats)
    probs_freq = torch.sigmoid(logits_freq).squeeze(-1).cpu().numpy()
    
    # Fuse
    fused = FUSION_WEIGHT_PIXEL * probs_pixel + FUSION_WEIGHT_FREQUENCY * probs_freq
    fused = np.clip(fused, 0.0, 1.0)
    
    # Aggregate per-frame scores
    final_score = confident_strategy(fused)
    return float(final_score)

# Run evaluation
results = []
for video_file, true_label in eval_clips:
    video_path = Path(DATA_DIR) / SAMPLE_PREFIX / video_file
    pred_score = analyze_video(video_path)
    pred_label = "FAKE" if pred_score > 0.5 else "REAL"
    correct = "Y" if pred_label == true_label else "N"
    results.append({
        "filename": video_file,
        "true_label": true_label,
        "pred_score": pred_score,
        "pred_label": pred_label,
        "correct": correct
    })
    print(f"  {video_file:20s} | {true_label:5s} | {pred_score:.4f} | {pred_label:5s} | {correct}")

# Print table
print()
print("-" * 70)
print(f"{'Filename':20s} | {'True':5s} | {'Score':6s} | {'Pred':5s} | {'OK':2s}")
print("-" * 70)
for r in results:
    print(f"{r['filename']:20s} | {r['true_label']:5s} | {r['pred_score']:.4f} | {r['pred_label']:5s} | {r['correct']}")

print("-" * 70)
total = len(results)
correct_count = sum(1 for r in results if r["correct"] == "Y")
accuracy = correct_count / total * 100
print(f"Accuracy: {correct_count}/{total} = {accuracy:.1f}%")

# AUC
try:
    from sklearn.metrics import roc_auc_score
    y_true = [1 if r["true_label"] == "FAKE" else 0 for r in results]
    y_score = [r["pred_score"] for r in results]
    auc = roc_auc_score(y_true, y_score)
    print(f"AUC:      {auc:.4f}")
except ImportError:
    print("AUC:      sklearn not installed, skipping")

print()
print("=" * 70)
print("END OF INFORMAL EVALUATION")
print("=" * 70)